# FX40 spectroscopy loader

Loads TIF files from the NX40 using their provided code and converts them to a dataframe

Make sure TIFF_Read_Write.py is in the same folder

TIFF_Read_Write.py found [here](https://spmdevhub.com/snippets/reading-and-writing-tiff-files/) on 23-09-2026

In [ ]:
import os
import numpy as np
from pspylib.tiff.reader import TiffReader
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from TIFF_Read_Write import Read_tiff

g_0 = 7.748e-5

In [ ]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\1.0V Area 1"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")
